# 부모 그래프와 서브그래프 — 정답본

원본: `src/solution/Chapter02/03-langgraph-advanced/01_subgraph.py`  
슬라이드: `docs/raw/Chapter02/day01/3-4-langgraph-advanced.pdf`

## 학습 목표

- SummaryState·ParentState의 경계
- 원본 코드를 CodeMate와 함께 읽고 실행 결과를 점검한다.


## 코드 흐름

fetch_document → summarize_document 안의 summarize·translate → respond

**읽을 때 확인할 점:** 서브그래프는 따로 컴파일한다. 자식 그래프의 summary만 부모 State로 복사된다.


## 실행 전 확인

요약, 번역, 최종 응답을 위해 LLM에 여러 번 접속한다.

아래 원본 코드 셀은 실행을 직접 선택해야 하는 `manual-run` 셀입니다. 실행 준비 셀은 프로젝트 루트를 찾고 원본 `.py`의 경로를 `__file__`에 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "Chapter02").is_dir()),
    None,
)
if _repo_root is None:
    raise FileNotFoundError("저장소 안에서 이 노트북을 열어 주세요.")
os.chdir(_repo_root)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root / "src"))
__file__ = str(_repo_root / "src/solution/Chapter02/03-langgraph-advanced/01_subgraph.py")


## 원본 코드

다음 셀에는 원본 `.py` 전체가 그대로 들어 있습니다.


In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()


# ── 서브그래프 ────────────────────────────────────────────────────────────────
# 서브그래프는 독립적으로 컴파일된 그래프로, 부모 그래프의 노드로 사용된다.
# 복잡한 워크플로우를 재사용 가능한 단위로 분리할 때 활용한다.

class SummaryState(TypedDict):
    """서브그래프 전용 State. 부모 State와 독립적이다."""
    text: str
    summary: str


def summarize_node(state: SummaryState) -> dict:
    '''
        이 노드가 하는 일은 원본 text (사용자의 입력 문서내용)을 요약하는것
        text에는 원본 데이터가 들어있고

        이 llm이 요약한 정보는 요약본이라는 상태에 별도로 저장
        summary -> 요약본 저장
        summary_subgraph -> 이 생태계에는 공유 가능한 상태 2가지
    '''
    result = llm.invoke(f'다음 텍스트를 한 문장으로 요약해줘:\n\n{state["text"]}')
    return {'summary': result.content}


def translate_node(state: SummaryState) -> dict:
    # '''
    #     지금은, 상태에 저장된 summary를 요약해 달라고 했다.
    #     근데 우리는 요약본 말고 원본을 영어로 번역해야 하는 상황이 있을 수도 있곘죠?

    #     summary_subgraph를 확장 한다 라고 한다면

    #         START -> supervisor node
    #                 (사용자가 요약해 달라고 했는가?)
    #                     y/n
    #                 /          \
    #             summarize       translate  (text 원본 기반 번역)
    #                 |              |
    #             translate   
    #                     \        /
    #                         END
    # '''
    result = llm.invoke(f'다음 문장을 영어로 번역해줘:\n\n{state["summary"]}')
    return {'summary': result.content}


summary_builder = StateGraph(SummaryState)
summary_builder.add_node('summarize', summarize_node)
summary_builder.add_node('translate', translate_node)
summary_builder.add_edge(START, 'summarize')
summary_builder.add_edge('summarize', 'translate')
summary_builder.add_edge('translate', END)

# 서브그래프를 독립적으로 컴파일
summary_subgraph = summary_builder.compile()


# result = summary_subgraph.invoke({'text': [HumanMessage(content='인공지능은 인간의 지능을 모방하는 컴퓨터 시스템으로, 최근 LLM의 등장으로 급격히 발전했습니다. 이제 AI는 텍스트 생성, 번역, 코딩 등 다양한 작업을 수행할 수 있습니다.')], 'summary': ''})
# print(result)

# ── 부모 그래프 ───────────────────────────────────────────────────────────────

class ParentState(TypedDict):
    messages: Annotated[list, add_messages]
    document: str
    summary: str


def fetch_document(state: ParentState) -> dict:
    # 실제로는 DB나 파일에서 문서를 가져온다.
    doc = '인공지능은 인간의 지능을 모방하는 컴퓨터 시스템으로, 최근 LLM의 등장으로 급격히 발전했습니다. 이제 AI는 텍스트 생성, 번역, 코딩 등 다양한 작업을 수행할 수 있습니다.'
    return {'document': doc}


def summarize_document(state: ParentState) -> dict:
    # 서브그래프를 노드처럼 호출
    result = summary_subgraph.invoke({'text': state['document'], 'summary': ''})
    return {'summary': result['summary']}


def respond_node(state: ParentState) -> dict:
    response = llm.invoke(
        f'문서 요약(영문): {state["summary"]}\n\n이를 바탕으로 사용자 질문에 답해줘: {state["messages"][-1].content}'
    )
    return {'messages': [response]}


parent_builder = StateGraph(ParentState)
parent_builder.add_node('fetch_document', fetch_document)
parent_builder.add_node('summarize_document', summarize_document)   # 서브그래프 노드
parent_builder.add_node('respond', respond_node)

parent_builder.add_edge(START, 'fetch_document')
parent_builder.add_edge('fetch_document', 'summarize_document')
parent_builder.add_edge('summarize_document', 'respond')
parent_builder.add_edge('respond', END)

app = parent_builder.compile()


result = app.invoke({
    'messages': [HumanMessage(content='문서의 핵심 내용이 뭐야?')],
    'document': '',
    'summary': '',
})

print(f'영문 요약: {result["summary"]}')
print(f'최종 답변: {result["messages"][-1].content}')


## CodeMate와 함께 검토

> 두 State의 필드를 비교하고 document가 영문 summary를 거쳐 최종 답변에 쓰이는 경로를 설명해 줘. 원본은 바꾸지 말고 검토 내용을 먼저 말해 줘.

제안된 변경은 어떤 문제를 해결하는지, 어떤 결과로 확인할지 설명해 보세요.

## 확인 질문

1. 부모와 자식 State를 나누는 이유는?
2. summarize_document가 반환하는 필드는?
